# 03c — Prompt + RAG hỏi đáp

## Mục tiêu
- Nối retriever + LLM thành chuỗi RAG.
- System prompt ép: chỉ trả lời theo ngữ cảnh, không bịa.
- So sánh câu hỏi **trong** tài liệu vs **ngoài** tài liệu.

Pipeline: câu hỏi → embedding → top-k chunk → nhét vào prompt → LLM.


## 1. Khởi tạo từ Chroma đã lưu + LLM


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path("../../").resolve()
load_dotenv(ROOT / ".env")

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("Thieu OPENAI_API_KEY trong .env")

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
# LangChain 1.x: cac helper chain nam trong langchain_classic
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

persist_dir = str(ROOT / "data" / "chroma_db")
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=OpenAIEmbeddings(),
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Retriever + LLM san sang.")


## 2. System prompt chặt + tạo chain


In [ ]:
system_prompt = (
    "Ban la tro ly AI huu ich. "
    "Su dung ngu canh duoi day de tra loi. "
    "Neu khong tim thay dap an, hay noi 'Toi khong biet', khong bia thong tin.\n\n"
    "Ngu canh:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, qa_chain)
print("RAG chain OK")


## 3. Câu hỏi trong tài liệu


In [ ]:
question = "Nhung cong cu nao duoc dung de xay dung chatbot RAG trong tai lieu?"
response = rag_chain.invoke({"input": question})
print("Cau hoi:", question)
print("\nTra loi:\n", response["answer"])
print("\nSo chunk context:", len(response.get("context", [])))


## 4. Câu hỏi ngoài tài liệu (kỳ vọng: không biết / không bịa)


In [ ]:
question2 = "Gia Bitcoin ngay mai la bao nhieu theo tai lieu nay?"
response2 = rag_chain.invoke({"input": question2})
print("Cau hoi:", question2)
print("\nTra loi:\n", response2["answer"])


## Bài tập nhỏ

1. Đổi `k` retrieve từ 3 → 5, hỏi lại cùng câu — câu trả lời có dài/chi tiết hơn không?
2. Hỏi về Level 1 của lộ trình AI — model có trích đúng ý không?
3. (Tuỳ chọn) Thử temperature=0.7 và quan sát độ "bay" của câu trả lời.


In [ ]:
# TODO: thu k=5
# retriever5 = vectorstore.as_retriever(search_kwargs={"k": 5})


## Checklist — đã hiểu khi…
- [ ] Vẽ được pipeline PDF → chunk → vector → retrieve → LLM
- [ ] Giải thích vì sao prompt cần câu "không bịa"
- [ ] Phân biệt được câu hỏi trong / ngoài corpus
- [ ] Sẵn sàng xem mỗi giai đoạn Jupyter như 1 module cho app Store sau này
